# **Downstream analysis of CS-CORE coexpression matrices**

This notebook analyzes CS-CORE-derived matrices across nine datasets to establish gene co-expression relationships.

In [ ]:
# Environment setup for CS-CORE co-expression downstream analysis

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors

In [ ]:
# Load all matrices

datasets_dict = {}

for i in range(1, 10):
    file_est = f'Dataset{i}_CSCORE_partial_matrix_est.csv' # contains coexpression coefficients
    file_p = f'Dataset{i}_CSCORE_partial_matrix_p.csv' # contains p-values
    
    df_est = pd.read_csv(file_est, index_col=0)
    df_p = pd.read_csv(file_p, index_col=0)
    
    # Store in a single structured dictionary
    datasets_dict[f'DS{i}'] = {
        'est': df_est,
        'p': df_p
    }
    print(f"DS{i}: Successfully loaded matrices from disk. Shape: {df_est.shape}")


# **Coexpression between two genes**

In [ ]:
def plot_coexpression_bar(gene_a, gene_b, datasets_integrated):
    """
    Plots a 3D-style horizontal bar plot displaying the co-expression values
    between two genes across nine single-cell datasets.
    """
    results = []
    
    # Unified publication color palette matching prior figures
    ds_colors = {
        'DS1': '#4CAF50', 'DS2': '#F4B86C', 'DS3': '#C96BF5',
        'DS4': '#7FFFD4', 'DS5': '#FF6161', 'DS6': '#6DE26A',
        'DS7': '#8A6CF4', 'DS8': '#4C9BE8', 'DS9': '#F46CB8'
    }

    # Extract estimates data matrices from the integrated dictionary structure
    for name, matrices in datasets_integrated.items():
        df_est = matrices['est']  # Target the co-expression estimates matrix
        
        # Check bidirectional presence in the matrix due to potential shape differences
        if gene_a in df_est.index and gene_b in df_est.columns:
            results.append({'Dataset': name, 'Correlation': float(df_est.at[gene_a, gene_b])})
        elif gene_b in df_est.index and gene_a in df_est.columns:
            results.append({'Dataset': name, 'Correlation': float(df_est.at[gene_b, gene_a])})

    if not results:
        print(f"Error: Pair {gene_a} <-> {gene_b} not found in the provided matrices.")
        return

    plot_df = pd.DataFrame(results)
    mean_val = plot_df['Correlation'].mean()
    plot_df['jitter'] = np.random.uniform(-0.06, 0.06, size=len(plot_df))

    # Visualization style setup
    sns.set_style("white")
    fig, ax = plt.subplots(figsize=(12, 4))

    # Maintain color scheme consistency (blue for positive coexpression, orange for negative)
    bar_color = '#FFB347' if mean_val > 0 else '#7AC0FF'

    # Multi-layered transparent rendering for the 3D-style bar effect
    for i in range(1, 15):
        ax.barh(0, mean_val, height=0.25, color=bar_color, alpha=0.06 * i, zorder=2)
    ax.barh(-0.01, mean_val, height=0.25, color='black', alpha=0.1, zorder=1)

    # Scatter individual dataset data points with localized jittering
    for _, row in plot_df.iterrows():
        ax.scatter(row['Correlation'], row['jitter'],
                   color=ds_colors.get(row['Dataset'], 'grey'), s=350,
                   edgecolor='black', linewidth=1.5, zorder=5, alpha=0.9)

    # Axis configuration and styling adjustments
    ax.axvline(0, color='#333333', linewidth=2, zorder=3)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-0.4, 0.4)
    
    ax.set_yticks([0])
    ax.set_yticklabels([f"{gene_a} ↔ {gene_b}"], fontsize=16, fontweight='bold', fontstyle='italic')
    
    ax.set_xticks(np.arange(-1, 1.1, 0.2))
    ax.tick_params(axis='x', labelsize=12)
    ax.grid(axis='x', linestyle='--', alpha=0.4)

    plt.title(f'Co-expression Analysis (Mean: {mean_val:.3f})', fontsize=18, fontweight='bold', pad=20)
    
    # Legend generation restricted to present datasets
    legend_handles = [plt.Line2D([0], [0], marker='o', color='w', label=k,
                      markerfacecolor=v, markersize=12, markeredgecolor='black') 
                      for k, v in ds_colors.items() if k in plot_df['Dataset'].values]
    ax.legend(handles=legend_handles, title='Datasets', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

    sns.despine(left=True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_coexpression_bar('MEF2A', 'P2RY12', datasets_dict)

MEF2A and P2RY12 strongly co-express. They are both key components of the homeostatic program.

In [ ]:
plot_coexpression_bar('FRMD4A', 'DPYD', datasets_dict)

FRMD4A and DPYD negatively co-express; FRMD4A is homeostatic, while DPYD is activation-related.

In [ ]:
plot_coexpression_bar('SPP1', 'TMEM163', datasets_dict)

SPP1 and TMEM163 have near-zero co-expression, suggesting they are likely unrelated to each other.

# **Finding top co-expression partners for a gene**

In [ ]:
def plot_target_top_correlations(datasets_integrated, target_gene, title_text=None, x_range=(-1, 1), min_datasets=3, n_pos=10, n_neg=10):
    """
    Identifies and plots the top positive and negative co-expression partners 
    for a target gene across the integrated datasets structure.
    """
    # 1. Data harvesting from the integrated dictionary structure
    all_correlations = []
    for ds_name, matrices in datasets_integrated.items():
        df_est = matrices['est']  # Target the co-expression estimates matrix
        
        if target_gene in df_est.index:
            # Extract correlation vector and drop self-correlation
            cor_vector = df_est.loc[target_gene].copy()
            if target_gene in cor_vector.index:
                cor_vector = cor_vector.drop(labels=[target_gene])
            
            # Clean technical artifacts where correlation is absolute 1.0
            cor_vector = cor_vector.apply(lambda x: 0 if abs(x) >= 0.999 else x)
            
            ds_df = cor_vector.reset_index()
            ds_df.columns = ['gene', f'Cor_{ds_name}']
            all_correlations.append(ds_df)

    if not all_correlations:
        print(f"Error: Target gene '{target_gene}' not found in any available matrices.")
        return

    # 2. Merge harvested correlation vectors into a single framework
    merged_df = all_correlations[0]
    for next_df in all_correlations[1:]:
        merged_df = pd.merge(merged_df, next_df, on='gene', how='outer')

    cor_cols = [c for c in merged_df.columns if c.startswith('Cor_')]
    merged_df['Presence_Count'] = merged_df[cor_cols].notna().sum(axis=1)
    
    # Filter by dataset replication reliability threshold
    filtered_df = merged_df[merged_df['Presence_Count'] >= min_datasets].copy()
    if filtered_df.empty: 
        print("No partners found passing the current dataset filtration thresholds.")
        return
    
    filtered_df['Mean_Cor'] = filtered_df[cor_cols].mean(axis=1)

    # 3. Select top upstream and downstream partners
    top_pos = filtered_df.sort_values(by='Mean_Cor', ascending=False).head(n_pos)
    top_neg = filtered_df.sort_values(by='Mean_Cor', ascending=True).head(n_neg)
    plot_data = pd.concat([top_neg, top_pos]).sort_values(by='Mean_Cor', ascending=True).reset_index(drop=True)

    # 4. Visualization engine setup
    sns.set_style("whitegrid")
    fig_height = max(6, len(plot_data) * 0.45)
    fig, ax = plt.subplots(figsize=(14, fig_height))

    # Maintain strict original color scheme definitions
    bar_pos, bar_neg = '#FFB347', '#7ac0ff'
    ds_palette = {
        'DS1': '#4CAF50', 'DS2': '#F4B86C', 'DS3': '#C96BF5',
        'DS4': '#7FFFD4', 'DS5': '#FF6161', 'DS6': '#6DE26A',
        'DS7': '#8A6CF4', 'DS8': '#4C9BE8', 'DS9': '#F46CB8'
    }

    y_pos = np.arange(len(plot_data))
    
    # Render heavy horizontal bars and individual dataset scatter points
    for idx, row in plot_data.iterrows():
        mean_val = row['Mean_Cor']
        ax.barh(y_pos[idx], mean_val, color=(bar_pos if mean_val > 0 else bar_neg),
                height=0.8, alpha=0.9, edgecolor='black', linewidth=0.8, zorder=2)

        for col in cor_cols:
            val = row[col]
            if not pd.isna(val):
                ds_id = col.replace('Cor_', '')
                ax.scatter(val, y_pos[idx] + np.random.uniform(-0.18, 0.18),
                           color=ds_palette.get(ds_id, 'grey'), s=180,
                           edgecolor='black', linewidth=0.8, zorder=5, alpha=0.9)

    # 5. Axis formatting and typography configurations
    ax.axvline(0, color='#333333', linewidth=1.5, zorder=3)
    ax.set_xlim(x_range)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_data['gene'].tolist(), fontsize=18, fontweight='bold', fontstyle='italic')
    ax.tick_params(axis='x', labelsize=18)
    
    header = title_text if title_text else f'Top Partners for {target_gene}'
    ax.set_title(header, fontsize=23, fontweight='bold', pad=25)
    ax.set_xlabel('Mean CS-CORE Correlation', fontsize=18, fontweight='bold')

    # Restricted legend generation to match structural layout rules
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w', label=k,
                                 markerfacecolor=v, markersize=12, markeredgecolor='black')
                       for k, v in ds_palette.items()]
    ax.legend(handles=legend_elements, title='Datasets', title_fontsize=18, fontsize=18, 
              bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, labelspacing=0.4)

    sns.despine(left=True)
    plt.tight_layout(pad=1.0)
    plt.show()

In [ ]:
plot_target_top_correlations(datasets_dict, 'SPP1', min_datasets=3)

SPP1, an activation-related gene, is strongly co-expressed with both the GPNMB+ signature (MYO1E, SLC11A1) and ribosomal-immune group genes (C1QB, HLAs, TMSB4X), while being negatively co-expressed with homeostatic genes (such as P2RY12, ARHGAP22, and FRMD4A).

In [ ]:
plot_target_top_correlations(datasets_dict, 'SRGAP2B', min_datasets=3)

SRGAP2B, a homeostatic gene, is strongly co-expressed with its evolutionary homolog SRGAP2 and other homeostatic genes, while showing negative co-expression with activation-related genes.

# **Establishing the structure of gene co-expression modules**

This section represents the core component of the analytical workflow. While cluster-based methods, such as hdWGCNA, identify presumptive gene co-expression modules that may lack perfect reproducibility, our framework allows us to directly quantify the connection strength between specific pairs or within entire groups of genes. This provides a more robust validation of the true underlying gene network structure.

In [ ]:
def plot_cscore_heatmap(datasets_integrated, gene_list, min_datasets=3):
    """
    Generates a publication-quality clustermap displaying the mean CS-CORE co-expression 
    coefficients for a pre-selected list of genes integrated across multiple datasets.
    """
    # 1. Extract and subset estimates matrices from the integrated structure
    sub_matrices = []
    for name, matrices in datasets_integrated.items():
        df_est = matrices['est']  # Target the co-expression estimates matrix
        
        present = [g for g in gene_list if g in df_est.index]
        if not present: 
            continue
            
        sub = df_est.loc[present, present].copy()
        # Clean technical artifacts where correlation is absolute 1.0
        sub = sub.apply(lambda x: x.map(lambda v: 0 if 0.999 < abs(v) < 1.0 else v))
        sub_matrices.append(sub)

    # Initialize empty dataframes matching the exact input gene list size
    full_idx = pd.Index(gene_list)
    sum_df = pd.DataFrame(0.0, index=full_idx, columns=full_idx)
    cnt_df = pd.DataFrame(0, index=full_idx, columns=full_idx)

    # Align and aggregate sub-matrices into a unified meta-matrix
    for sub in sub_matrices:
        aligned = sub.reindex(index=full_idx, columns=full_idx)
        mask = aligned.notna()
        sum_df[mask] += aligned.fillna(0)
        cnt_df[mask] += 1

    # Calculate global cross-dataset mean values
    heatmap_data = (sum_df / cnt_df).fillna(0) 

    # 2. Configure publication-ready visualization styling definitions
    plt.rcParams['figure.dpi'] = 300
    plt.rcParams['lines.linewidth'] = 2.5
    
    # Custom scientific divergence color palette (Blue-White-Orange)
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#1C27EA", "#FFFFFF", "#FF8500"])

    # 3. Render Clustermap framework
    size = 14
    g = sns.clustermap(
        heatmap_data,
        method='average',
        cmap=cmap,
        vmin=-0.5, vmax=0.5, center=0,
        linewidths=0.3,
        linecolor='#D0D0D0',
        dendrogram_ratio=0.1,
        figsize=(size, size), 
        cbar_pos=None
    )

    # 4. Resolve the absolute ordered gene sequence directly from the clustered framework
    reordered_ind = g.dendrogram_row.reordered_ind
    ordered_genes_list = [heatmap_data.index[i] for i in reordered_ind]

    # 5. Axis tick labels formatting and absolute coordinate mapping
    g.ax_heatmap.set_xticks(np.arange(len(ordered_genes_list)) + 0.5)
    g.ax_heatmap.set_yticks(np.arange(len(ordered_genes_list)) + 0.5)
    
    g.ax_heatmap.set_xticklabels(ordered_genes_list, fontsize=12, rotation=90, 
                                 fontstyle='italic', fontweight='bold')
    g.ax_heatmap.set_yticklabels(ordered_genes_list, fontsize=12, rotation=0, 
                                 fontstyle='italic', fontweight='bold')

    plt.subplots_adjust(bottom=0.2, right=0.85)

    # 6. Integrated horizontal colorbar configuration
    cax = g.fig.add_axes([0.13, 0.05, 0.35, 0.02]) 
    cb = plt.colorbar(g.ax_heatmap.collections[0], cax=cax, orientation='horizontal')
    cb.ax.tick_params(labelsize=12)
    cb.outline.set_linewidth(1.5)
    cb.set_label('Mean Correlation', fontsize=12, fontweight='bold')

    # 7. Final cosmetic layout polish
    g.ax_heatmap.set_facecolor('#ECECEC')
    for _, spine in g.ax_heatmap.spines.items():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(2.0)

    g.ax_heatmap.set_xlabel("")
    g.ax_heatmap.set_ylabel("")

    plt.show()
    return ordered_genes_list

In [ ]:
genes_small = ['GPNMB', 'IQGAP2', 'DPYD', 'RPL23', 'TMSB10', 'SPP1', 'P2RY12', 'MEF2A', 'FRMD4A', 'HSPB1']

heatmap = plot_cscore_heatmap(datasets_dict, genes_small)

In [ ]:
genes = [
    'IL6ST', 'ELMO1', 'IQGAP2', 'HLA-C', 'HSPA1A', 'FOSL2', 'MECOM', 'REL', 'IL1B', 'ARHGAP22',
    'HSPH1', 'ETV6', 'ARHGAP26', 'TANC2', 'FRMD4A', 'NAV3', 'FMN1', 'ARHGAP15', 'DPYD', 'DNAJB1',
    'SPTLC2', 'TMEM163', 'KCNMA1', 'RPS11', 'APOE', 'FMNL2', 'PPARG', 'OOEP', 'SORL1', 'RPL19',
    'SPP1', 'MITF', 'RPS6', 'RPL13A', 'RPL23', 'HIF1A', 'SLC11A1', 'SRGAP2', 'HLA-DRA', 'CX3CR1',
    'SRGAP2B', 'STARD13', 'C1QB', 'MEF2C', 'RPLP1', 'TMSB10', 'PTPRG', 'GPNMB', 'FKBP5', 'ITPR2',
    'MEF2A', 'RPS27A', 'TMSB4X', 'ANKRD44', 'HLA-B', 'DOCK8', 'MYO1E', 'RPL13', 'NHSL1', 'FTL',
    'LRRK2', 'P2RY12', 'HSPB1', 'FOXP1'
]

heatmap = plot_cscore_heatmap(datasets_dict, genes)

We obtain the exact same heatmap as presented in the article. Homeostatic genes cluster together and negatively co-express with other groups. Ribosomal and immune genes, HSPs, and the GPNMB+ signature along with inflammatory genes roughly correspond to the three major activation-related gene groups.

# **Validating statistical significance**

We can use p-value tables to obtain information about the statistical significance of each pair, and the matrix as a whole. These p-values were obtained from a ~25,000 × 25,000 comparison for each dataset, followed by FDR multiple test correction, but are still usually very low due to the large number of cells in each dataset.

In [ ]:
def plot_dataset_p_value_heatmap(dataset_num, gene_list, output_dpi=300):
    """
    Generates and saves a high-resolution log-scaled adjusted P-value heatmap 
    for a specific cross-species single-cell dataset.
    """
    import os
    
    file_path = f'Dataset{dataset_num}_CSCORE_partial_matrix_p.csv'
    
    # Check if the requested file exists before attempting to read
    if not os.path.exists(file_path):
        print(f"Skipping DS{dataset_num}: File {file_path} not found.")
        return
        
    df_p = pd.read_csv(file_path, index_col=0)
    full_idx = pd.Index(gene_list)
    
    # 1. Align matrix architecture to match the fixed global gene list ordering
    p_matrix = df_p.reindex(index=full_idx, columns=full_idx)
    
    # 2. Mathematical transformation for log-scaled color gradient mapping
    plot_data = p_matrix.copy()
    plot_data = plot_data.replace(0.0, 1e-100)  # Avoid log10 infinity runtime errors
    neg_log_p = -np.log10(plot_data)
    
    # 3. Structural text annotations generation for non-significant cells
    def get_annotation(val):
        if pd.isna(val):
            return ""
        if val > 0.05:
            return "ns"
        return ""
    annot_matrix = p_matrix.applymap(get_annotation)
    
    # 4. Generate visual masking layer to maintain clean white cells for missing entities
    mask = p_matrix.isna()
    
    # 5. Canvas orchestration and rendering engine execution
    plt.figure(figsize=(14, 12), dpi=output_dpi)
    
    sns.heatmap(
        neg_log_p, 
        mask=mask,          
        cmap='YlOrRd',    
        vmin=0, vmax=100,    
        cbar_kws={'label': '-log10(P-value adj)'},  # Standard clean text string
        linewidths=0.5, 
        linecolor='gray',
        annot=annot_matrix, 
        annot_kws={"size": 8, "color": "black", "weight": "bold"}, 
        fmt=''             
    )
    
    # Axis formatting layout definitions
    plt.title(f'DS{dataset_num} Significance Matrix Framework', fontsize=16, fontweight='bold', pad=20)
    plt.xticks(rotation=90, fontsize=9, fontweight='bold', fontstyle='italic')
    plt.yticks(rotation=0, fontsize=9, fontweight='bold', fontstyle='italic')
    
    plt.tight_layout()

In [ ]:
# Fixed target sequence list of pre-selected genes
ordered_genes = [
    'TMSB10', 'TMSB4X', 'FTL', 'RPS27A', 'RPL23', 'RPL19', 'RPS11', 'RPS6',
    'RPLP1', 'RPL13A', 'RPL13', 'APOE', 'C1QB', 'HLA-DRA', 'HLA-C', 'HLA-B',
    'MECOM', 'OOEP', 'IL6ST', 'CX3CR1', 'P2RY12', 'ITPR2', 'NAV3', 'MEF2C',
    'SRGAP2', 'SRGAP2B', 'DOCK8', 'MEF2A', 'FRMD4A', 'ARHGAP22', 'SORL1',
    'ANKRD44', 'HSPA1A', 'HSPB1', 'HSPH1', 'DNAJB1', 'ELMO1', 'ARHGAP15',
    'LRRK2', 'FOXP1', 'FKBP5', 'NHSL1', 'TANC2', 'KCNMA1', 'MITF', 'GPNMB',
    'DPYD', 'FMN1', 'IQGAP2', 'STARD13', 'PTPRG', 'FMNL2', 'PPARG', 'MYO1E',
    'TMEM163', 'SPTLC2', 'ETV6', 'ARHGAP26', 'IL1B', 'SPP1', 'SLC11A1',
    'FOSL2', 'REL', 'HIF1A'
]

# Run processing sequentially for all 9 cross-species datasets
for i in range(1, 10):
    plot_dataset_p_value_heatmap(dataset_num=i, gene_list=ordered_genes)
    plt.show() 

As can be inferred from the matrices, most co-expression interactions within the groups are highly significant. DS6 contains multiple empty cells, likely due to the incomplete annotation of the Macaca fascicularis genome.